In [ ]:
import glob, os, shutil, subprocess, tempfile
from pathlib import Path
OCTAVE = shutil.which("octave-cli") or shutil.which("octave")
if OCTAVE is None and os.name == "nt":
    candidates = glob.glob(r"C:\Program Files\GNU Octave\Octave-*\mingw64\bin\octave-cli.exe")
    if candidates:
        OCTAVE = sorted(candidates)[-1]
if OCTAVE is None:
    raise RuntimeError("Octave CLI not found. Add octave-cli to PATH or install GNU Octave.")
WORK = Path(tempfile.mkdtemp(prefix="me256-coursework-"))
print(f"Octave: {OCTAVE}")
print(f"Temporary source directory: {WORK}")

for relative in ['vmso3', 'vmso31']:
    (WORK / relative).mkdir(parents=True, exist_ok=True)


# ME256 Homework 3: variational models for flexible structures

**Official statement:** [ME256 Homework 3 (2011)](https://mecheng.iisc.ac.in/suresh/me256/hw2011/homework3.html). The outline below paraphrases the tasks for study; consult the linked page for the complete statement and exact requirements.

The assignment has four modeling questions:

1. Formulate a variational model for a flexible wire of fixed length $L$, initially on the horizontal axis with one endpoint at the origin and the other at $(L,0)$. The free endpoint is moved vertically to $(L,L)$. Identify a suitable function space and norm, check the norm axioms, and discuss what can be said about the solution.
2. Extend the wire model to a flexible cloth that bends but does not stretch; outline the formulation, with a more complete derivation as an optional extension.
3. Construct a sufficiently motivated variational example that uses an inner product, not only a norm.
4. Formulate a structural-optimization problem in a variational framework and explain the function space and norm or inner product needed for its functionals.

The archived MATLAB code is relevant chiefly to the first two topics: it discretizes the wire and cloth configurations, measures deviation from a reference state, and uses equality constraints to approximate fixed segment or mesh-edge lengths. The last two questions are open-ended modeling work and are not answered by these code listings alone.

## Study prompts

- Identify the continuous configuration and admissible function space suggested by the discrete variables, and verify the norm properties if a norm is proposed.
- Explain how the length-preservation constraints approximate inextensibility, then consider what changes as the cloth mesh is refined.
- Develop separate examples for an inner-product-based functional and a structural-optimization functional.
- Relate numerical objectives and constraints back to the modeling assumptions in the official statement.

## Reproducibility and version history

This notebook uses a Python 3 Jupyter kernel to materialize the embedded MATLAB source into a temporary directory, then invokes Octave CLI. The `.m` sources are embedded below; they are not stored separately in this repository. Install Jupyter/IPython and an Octave installation with the `optim` package. MATLAB with Optimization Toolbox remains the closest match to the original runtime. The variant code PDFs have matching filenames but different contents, so both versions are embedded and tested separately. In `vmso31`, one edge direction is absent; that variant is preserved rather than silently repaired.

## Variant `vmso3`

This version uses the coarser cloth discretization. Its constraint routine includes horizontal and vertical edges and both diagonals in each cell.

## Variant `vmso31`

This later-looking folder has a finer cloth discretization but a different set of distance constraints. Preserve it as a separate historical variant; do not assume the names alone indicate which copy is authoritative.

### Separate improvement ideas from historical code

A useful follow-up study would compare mesh-refinement behavior and test whether the intended cloth constraints are complete. Keep such experiments in new files or a new variant so the reconstructed originals remain traceable.

## Embedded source code

Each following code cell contains one MATLAB source file and writes it into a temporary working directory. Run the notebook from top to bottom; no reconstructed `.m` files are required in the repository.

### `vmso3/wire2d_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'wire2d_simulation.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize a segmented wire configuration and plot its final shape.
clc;close all;clear all;
global segment_count segment_length initial_x_positions endpoint_vertical_position
wire_length=1;
segment_count=40;
segment_length=wire_length/segment_count;
endpoint_vertical_position=wire_length;
initial_x_positions=zeros(1,segment_count+1);
for point_index=1:1:segment_count+1
    initial_x_positions(point_index)=(point_index-1)*wire_length/segment_count;
end
initial_segment_angles=zeros(segment_count+1,1);
[optimal_segment_angles,objective_value,optimizer_exit_flag,solver_output]=fmincon(@wire2d,initial_segment_angles,[],[],[],[],zeros(segment_count+1,1),ones(segment_count+1,1)*2*pi,[],optimset('Algorithm','active-set'));
fprintf('Wire objective %.6g; optimizer exit flag %d\n', objective_value, optimizer_exit_flag);
x_coordinates=zeros(1,segment_count+1);
y_coordinates=zeros(1,segment_count+1);
x_coordinates(segment_count+1)=initial_x_positions(segment_count+1);
y_coordinates(segment_count+1)=endpoint_vertical_position;
for segment_index=segment_count:-1:1
    x_coordinates(segment_index)=x_coordinates(segment_index+1)-segment_length*cos(optimal_segment_angles(segment_index));
    y_coordinates(segment_index)=y_coordinates(segment_index+1)-segment_length*sin(optimal_segment_angles(segment_index));
end
plot(x_coordinates,y_coordinates)
hold on
plot(initial_x_positions,zeros(1,segment_count+1),'r')
axis image
legend('Final Configuration','Initial Configuration','Location','NorthEastOutside')
title('Wire Simulation')


### `vmso3/wire2d.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'wire2d.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize the wire's total squared displacement from its start.
function displacement_norm=wire2d(segment_angles)
    global segment_count segment_length initial_x_positions endpoint_vertical_position
    x_coordinates=zeros(1,segment_count+1);
    y_coordinates=zeros(1,segment_count+1);
    x_coordinates(segment_count+1)=initial_x_positions(segment_count+1);
    y_coordinates(segment_count+1)=endpoint_vertical_position;
    for segment_index=segment_count:-1:1
        x_coordinates(segment_index)=x_coordinates(segment_index+1)-segment_length*cos(segment_angles(segment_index));
        y_coordinates(segment_index)=y_coordinates(segment_index+1)-segment_length*sin(segment_angles(segment_index));
    end
    displacement_norm=0;
    for point_index=segment_count+1:-1:1
        displacement_norm=displacement_norm+((x_coordinates(point_index)-initial_x_positions(point_index))^2+y_coordinates(point_index)^2);
    end
    displacement_norm=sqrt(displacement_norm);


### `vmso3/cloth_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'cloth_simulation.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize and plot a triangulated cloth mesh with fixed edge lengths.
clc;close all;clear all;
global mesh_subdivisions mesh_edge_length initial_x_grid initial_y_grid corner_height column_indices row_indices
cloth_length=1;
mesh_subdivisions=3;
mesh_edge_length=cloth_length/mesh_subdivisions;
corner_height=cloth_length;
initial_x_grid=zeros(1,mesh_subdivisions+1);
initial_y_grid=zeros(1,mesh_subdivisions+1);
column_indices=zeros(1,(mesh_subdivisions+1)*(mesh_subdivisions+1));
row_indices=zeros(1,(mesh_subdivisions+1)*(mesh_subdivisions+1));
for outer_grid_index=1:1:mesh_subdivisions+1
    for inner_grid_index=1:1:mesh_subdivisions+1
        column_indices((outer_grid_index-1)*(mesh_subdivisions+1)+inner_grid_index)=inner_grid_index;
        row_indices((outer_grid_index-1)*(mesh_subdivisions+1)+inner_grid_index)=outer_grid_index;
    end
end
for outer_grid_index=1:1:mesh_subdivisions+1
    initial_x_grid(outer_grid_index)=(outer_grid_index-1)*cloth_length/mesh_subdivisions;
    initial_y_grid(outer_grid_index)=(outer_grid_index-1)*cloth_length/mesh_subdivisions;
end
% Pin the far corner to its prescribed displaced position.
fixed_node_constraint_matrix=zeros(3,3*(mesh_subdivisions+1)^2);
fixed_node_constraint_matrix(1,(mesh_subdivisions+1)^2)=1;
fixed_node_constraint_matrix(2,2*(mesh_subdivisions+1)^2)=1;
fixed_node_constraint_matrix(3,3*(mesh_subdivisions+1)^2)=1;
fixed_node_constraint_values=[initial_x_grid(mesh_subdivisions+1);initial_y_grid(mesh_subdivisions+1);corner_height];
initial_node_coordinates=[(column_indices-1)*cloth_length/mesh_subdivisions, (row_indices-1)*cloth_length/mesh_subdivisions, ones(1,(mesh_subdivisions+1)*(mesh_subdivisions+1))*corner_height];
[optimized_node_coordinates,objective_value,optimizer_exit_flag,solver_output]=fmincon(@cloth,initial_node_coordinates,[],[],fixed_node_constraint_matrix,fixed_node_constraint_values,[],[],@dist,optimset('Algorithm','interior-point'));
fprintf('Cloth objective %.6g; optimizer exit flag %d\n', objective_value, optimizer_exit_flag);
% The optimizer vector stores every node x-coordinate, then y, then z.
x_coordinates=optimized_node_coordinates(1:(mesh_subdivisions+1)^2);
y_coordinates=optimized_node_coordinates((mesh_subdivisions+1)^2+1:2*(mesh_subdivisions+1)^2);
z_coordinates=optimized_node_coordinates(2*(mesh_subdivisions+1)^2+1:3*(mesh_subdivisions+1)^2);
for outer_grid_index=1:1:mesh_subdivisions
    for inner_grid_index=1:1:mesh_subdivisions
        patch([x_coordinates(inner_grid_index+(outer_grid_index-1)*(mesh_subdivisions+1));x_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));x_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1))],...
        [y_coordinates(inner_grid_index+(outer_grid_index-1)*(mesh_subdivisions+1));y_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));y_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1))],...
        [z_coordinates(inner_grid_index+(outer_grid_index-1)*(mesh_subdivisions+1));z_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));z_coordinates(inner_grid_index+outer_grid_index* (mesh_subdivisions+1))],'b','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.4)
        end
        patch([x_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));x_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1));x_coordinates(inner_grid_index+1+outer_grid_index*(mesh_subdivisions+1))],...
        [y_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));y_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1));y_coordinates(inner_grid_index+1+outer_grid_index*(mesh_subdivisions+1))],...
        [z_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));z_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1));z_coordinates(inner_grid_index+1+outer_grid_index* (mesh_subdivisions+1))],'r','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.4)
        end

    end
end
grid on


### `vmso3/cloth.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'cloth.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize mesh-node displacement from the initial cloth plane.
function displacement_norm=cloth(node_coordinates)
    global mesh_subdivisions initial_x_grid initial_y_grid column_indices row_indices
    displacement_norm=0;
    for node_index=1:1:(mesh_subdivisions+1)*(mesh_subdivisions+1)
        displacement_norm=displacement_norm+((node_coordinates(node_index)-initial_x_grid(column_indices(node_index)))^2+(node_coordinates(node_index+(mesh_subdivisions+1)*(mesh_subdivisions+1))-initial_y_grid(row_indices(node_index)))^2+(node_coordinates(node_index+2*(mesh_subdivisions+1)*(mesh_subdivisions+1))) ^2);
    end
    displacement_norm=sqrt(displacement_norm);


### `vmso3/dist.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'dist.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Equality constraints preserve horizontal, vertical, and diagonal mesh lengths.
function [c,ceq]=dist(node_coordinates)
    global mesh_subdivisions mesh_edge_length
    ceq=zeros(2*mesh_subdivisions*(mesh_subdivisions+1)+2*mesh_subdivisions^2,1);
    constraint_index=1;
    for outer_grid_index=1:1:mesh_subdivisions+1
        for inner_grid_index=1:1:mesh_subdivisions
            ceq(constraint_index)=sqrt((node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)*(mesh_subdivisions+1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)*(mesh_subdivisions+1)))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)*(mesh_subdivisions+1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)*(mesh_subdivisions+1)))^2)- mesh_edge_length;
            constraint_index=constraint_index+1;
        end
    end
    for inner_grid_index=1:1:mesh_subdivisions+1
        for outer_grid_index=1:1:mesh_subdivisions
            ceq(constraint_index)=sqrt((node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index+(mesh_subdivisions+1)*(mesh_subdivisions+1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)*(mesh_subdivisions+1)))^2+...
            (node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index+2*(mesh_subdivisions+1)*(mesh_subdivisions+1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)*(mesh_subdivisions+1)))^2)-mesh_edge_length;
            constraint_index=constraint_index+1;
        end
    end
    % Preserve the source constraint ordering, including the single-step diagonal index advance.
    for outer_grid_index=1:1:mesh_subdivisions
        for inner_grid_index=1:1:mesh_subdivisions
            ceq(constraint_index)=sqrt((node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*outer_grid_index)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*outer_grid_index+(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)^2))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*outer_grid_index+2*(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)^2))^2)-sqrt(2)*mesh_edge_length;
            ceq(constraint_index+1)=sqrt((node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index)-node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index+(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)^2))^2+...
            (node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index+2*(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)^2))^2)-sqrt(2)*mesh_edge_length;
            constraint_index=constraint_index+1;
        end
    end
    c=[];


### `vmso31/wire2d_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'wire2d_simulation.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize a segmented wire configuration and plot its final shape.
clc;close all;clear all;
global segment_count segment_length initial_x_positions endpoint_vertical_position
wire_length=1;
segment_count=40;
segment_length=wire_length/segment_count;
endpoint_vertical_position=wire_length;
initial_x_positions=zeros(1,segment_count+1);
for point_index=1:1:segment_count+1
    initial_x_positions(point_index)=(point_index-1)*wire_length/segment_count;
end
initial_segment_angles=zeros(segment_count+1,1);
[optimal_segment_angles,objective_value,optimizer_exit_flag,solver_output]=fmincon(@wire2d,initial_segment_angles,[],[],[],[],zeros(segment_count+1,1),ones(segment_count+1,1)*2*pi,[],optimset('Algorithm','active-set'));
fprintf('Wire objective %.6g; optimizer exit flag %d\n', objective_value, optimizer_exit_flag);
x_coordinates=zeros(1,segment_count+1);
y_coordinates=zeros(1,segment_count+1);
x_coordinates(segment_count+1)=initial_x_positions(segment_count+1);
y_coordinates(segment_count+1)=endpoint_vertical_position;
for segment_index=segment_count:-1:1
    x_coordinates(segment_index)=x_coordinates(segment_index+1)-segment_length*cos(optimal_segment_angles(segment_index));
    y_coordinates(segment_index)=y_coordinates(segment_index+1)-segment_length*sin(optimal_segment_angles(segment_index));
end
plot(x_coordinates,y_coordinates)
hold on
plot(initial_x_positions,zeros(1,segment_count+1),'r')
axis image
legend('Final Configuration','Initial Configuration','Location','NorthEastOutside')
title('Wire Simulation')


### `vmso31/wire2d.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'wire2d.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize the wire's total squared displacement from its start.
function displacement_norm=wire2d(segment_angles)
    global segment_count segment_length initial_x_positions endpoint_vertical_position
    x_coordinates=zeros(1,segment_count+1);
    y_coordinates=zeros(1,segment_count+1);
    x_coordinates(segment_count+1)=initial_x_positions(segment_count+1);
    y_coordinates(segment_count+1)=endpoint_vertical_position;
    for segment_index=segment_count:-1:1
        x_coordinates(segment_index)=x_coordinates(segment_index+1)-segment_length*cos(segment_angles(segment_index));
        y_coordinates(segment_index)=y_coordinates(segment_index+1)-segment_length*sin(segment_angles(segment_index));
    end
    displacement_norm=0;
    for point_index=segment_count+1:-1:1
        displacement_norm=displacement_norm+((x_coordinates(point_index)-initial_x_positions(point_index))^2+y_coordinates(point_index)^2);
    end
    displacement_norm=sqrt(displacement_norm);


### `vmso31/cloth_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'cloth_simulation.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize and plot a triangulated cloth mesh with distance constraints.
clc;close all;clear all;
global mesh_subdivisions mesh_edge_length initial_x_grid initial_y_grid corner_height column_indices row_indices
cloth_length=1;
mesh_subdivisions=10;
mesh_edge_length=cloth_length/mesh_subdivisions;
corner_height=cloth_length;
initial_x_grid=zeros(1,mesh_subdivisions+1);
initial_y_grid=zeros(1,mesh_subdivisions+1);
column_indices=zeros(1,(mesh_subdivisions+1)*(mesh_subdivisions+1));
row_indices=zeros(1,(mesh_subdivisions+1)*(mesh_subdivisions+1));
for outer_grid_index=1:1:mesh_subdivisions+1
    for inner_grid_index=1:1:mesh_subdivisions+1
        column_indices((outer_grid_index-1)*(mesh_subdivisions+1)+inner_grid_index)=inner_grid_index;
        row_indices((outer_grid_index-1)*(mesh_subdivisions+1)+inner_grid_index)=outer_grid_index;
    end
end
for outer_grid_index=1:1:mesh_subdivisions+1
    initial_x_grid(outer_grid_index)=(outer_grid_index-1)*cloth_length/mesh_subdivisions;
    initial_y_grid(outer_grid_index)=(outer_grid_index-1)*cloth_length/mesh_subdivisions;
end
% Pin the far corner to its prescribed displaced position.
fixed_node_constraint_matrix=zeros(3,3*(mesh_subdivisions+1)^2);
fixed_node_constraint_matrix(1,(mesh_subdivisions+1)^2)=1;
fixed_node_constraint_matrix(2,2*(mesh_subdivisions+1)^2)=1;
fixed_node_constraint_matrix(3,3*(mesh_subdivisions+1)^2)=1;
fixed_node_constraint_values=[initial_x_grid(mesh_subdivisions+1);initial_y_grid(mesh_subdivisions+1);corner_height];
initial_node_coordinates=[(column_indices-1)*cloth_length/mesh_subdivisions, (row_indices-1)*cloth_length/mesh_subdivisions, ones(1,(mesh_subdivisions+1)*(mesh_subdivisions+1))*corner_height];
[optimized_node_coordinates,objective_value,optimizer_exit_flag,solver_output]=fmincon(@cloth,initial_node_coordinates,[],[],fixed_node_constraint_matrix,fixed_node_constraint_values,[],[],@dist,optimset('Algorithm','interior-point'));
fprintf('Cloth objective %.6g; optimizer exit flag %d\n', objective_value, optimizer_exit_flag);
% The optimizer vector stores every node x-coordinate, then y, then z.
x_coordinates=optimized_node_coordinates(1:(mesh_subdivisions+1)^2);
y_coordinates=optimized_node_coordinates((mesh_subdivisions+1)^2+1:2*(mesh_subdivisions+1)^2);
z_coordinates=optimized_node_coordinates(2*(mesh_subdivisions+1)^2+1:3*(mesh_subdivisions+1)^2);
for outer_grid_index=1:1:mesh_subdivisions
    for inner_grid_index=1:1:mesh_subdivisions
        patch([x_coordinates(inner_grid_index+(outer_grid_index-1)*(mesh_subdivisions+1));x_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));x_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1))],...
        [y_coordinates(inner_grid_index+(outer_grid_index-1)*(mesh_subdivisions+1));y_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));y_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1))],...
        [z_coordinates(inner_grid_index+(outer_grid_index-1)*(mesh_subdivisions+1));z_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));z_coordinates(inner_grid_index+outer_grid_index* (mesh_subdivisions+1))],'b','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.5)
        end
        patch([x_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));x_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1));x_coordinates(inner_grid_index+1+outer_grid_index*(mesh_subdivisions+1))],...
        [y_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));y_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1));y_coordinates(inner_grid_index+1+outer_grid_index*(mesh_subdivisions+1))],...
        [z_coordinates(inner_grid_index+1+(outer_grid_index-1)*(mesh_subdivisions+1));z_coordinates(inner_grid_index+outer_grid_index*(mesh_subdivisions+1));z_coordinates(inner_grid_index+1+outer_grid_index* (mesh_subdivisions+1))],'r','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.5)
        end

    end
end
grid on


### `vmso31/cloth.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'cloth.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize mesh-node displacement from the initial cloth plane.
function displacement_norm=cloth(node_coordinates)
    global mesh_subdivisions initial_x_grid initial_y_grid column_indices row_indices
    displacement_norm=0;
    for node_index=1:1:(mesh_subdivisions+1)*(mesh_subdivisions+1)
        displacement_norm=displacement_norm+((node_coordinates(node_index)-initial_x_grid(column_indices(node_index)))^2+(node_coordinates(node_index+(mesh_subdivisions+1)*(mesh_subdivisions+1))-initial_y_grid(row_indices(node_index)))^2+(node_coordinates(node_index+2*(mesh_subdivisions+1)*(mesh_subdivisions+1))) ^2);
    end
    displacement_norm=sqrt(displacement_norm);


### `vmso31/dist.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'dist.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Original variant constraints: one edge direction and both cell diagonals.
function [c,ceq]=dist(node_coordinates)
    global mesh_subdivisions mesh_edge_length
    ceq=zeros(mesh_subdivisions*(mesh_subdivisions+1)+2*mesh_subdivisions^2,1);
    constraint_index=1;
    for outer_grid_index=1:1:mesh_subdivisions+1
        for inner_grid_index=1:1:mesh_subdivisions
            ceq(constraint_index)=sqrt((node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)*(mesh_subdivisions+1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)*(mesh_subdivisions+1)))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)*(mesh_subdivisions+1))-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)*(mesh_subdivisions+1)))^2)- mesh_edge_length;
            constraint_index=constraint_index+1;
        end
    end
    % Preserve the source constraint ordering, including the single-step diagonal index advance.
    for outer_grid_index=1:1:mesh_subdivisions
        for inner_grid_index=1:1:mesh_subdivisions
            ceq(constraint_index)=sqrt((node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*outer_grid_index)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*outer_grid_index+(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)^2))^2+...
            (node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*outer_grid_index+2*(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)^2))^2)-sqrt(2)*mesh_edge_length;
            ceq(constraint_index+1)=sqrt((node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index)-node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)))^2+...
            (node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index+(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+(mesh_subdivisions+1)^2))^2+...
            (node_coordinates(inner_grid_index+(mesh_subdivisions+1)*outer_grid_index+2*(mesh_subdivisions+1)^2)-node_coordinates(inner_grid_index+1+(mesh_subdivisions+1)*(outer_grid_index-1)+2*(mesh_subdivisions+1)^2))^2)-sqrt(2)*mesh_edge_length;
            constraint_index=constraint_index+1;
        end
    end
    c=[];


## Octave 11.3.0 test results

Both wire scripts execute, but Octave `optim` returns exit flag 0 at its 100-iteration limit, so their runs did not converge. Both cloth scripts fail in Octave `fmincon` with a singular-matrix warning followed by “could not regain binding constraints.” This is recorded rather than hidden or worked around by changing the optimizer or constraint model. The printed input configuration satisfies the prescribed length equalities; the optimizer failure needs comparison with MATLAB and review of the original formulation. The two variants remain distinct.

Run statuses and stderr are captured in the cell output below.

In [ ]:
import subprocess

def run_case(label, folder, script, needs_optim=True):
    setup = "set(0,'defaultfigurevisible','off');"
    if needs_optim:
        setup += "pkg load optim;"
    setup += "if any(strcmp(available_graphics_toolkits(),'gnuplot')), graphics_toolkit('gnuplot'); end;"
    setup += f"run('{script}');"
    result = subprocess.run(
        [OCTAVE, "--no-gui", "--quiet", "--eval", setup],
        cwd=WORK / folder, text=True, capture_output=True, errors="replace"
    )
    status = "PASS" if result.returncode == 0 else "FAILED"
    print(f"\n=== {label}: {status} (exit code {result.returncode}) ===")
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    return result.returncode

run_case('vmso3 wire', 'vmso3', 'wire2d_simulation.m', True)
run_case('vmso3 cloth', 'vmso3', 'cloth_simulation.m', True)
run_case('vmso31 wire', 'vmso31', 'wire2d_simulation.m', True)
run_case('vmso31 cloth', 'vmso31', 'cloth_simulation.m', True)



=== vmso3 wire: PASS (exit code 0) ===
Wire objective 4.34721; optimizer exit flag 0
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communication with gnuplot
uses a one-directional pipe and limited information is passed back to the
Octave interpreter so most changes made interactively in the plot window
will not be reflected in the graphics properties managed by Octave.  For
example, if the plot window is closed with a mouse click, Octave will not
be notified and will not update its internal list of open figure windows.
The qt toolkit is recommended instead.

The FLTK graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.
The qt toolkit is recommended instead.

=== vmso3 cloth: FAILED (exit code 1) ===
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communica

In [ ]:
import shutil
shutil.rmtree(WORK)
print("Temporary generated source files removed.")